<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/02-column-expressions.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 3.2 — Column expressions: col(), lit(), expr(), when/otherwise

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, expr
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("3.2-column-expressions")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## A Column is a free-floating expression tree — reusable across DataFrames

In [ ]:
line_total = col("quantity") * col("unit_price")   # no DataFrame involved
print(type(line_total), "\n", line_total)           # printable tree, no data

orders.select("order_id", line_total.alias("total")).show(3)

In [ ]:
# Errors surface at ANALYSIS, not execution — no action needed to trip this:
try:
    orders.select(col("quantiy") * 2)
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

## lit(): constants must be expressions too

In [ ]:
orders.withColumn("source", lit("web")).select("order_id", "source").show(3)

try:
    orders.withColumn("source", "web")   # the classic mistake
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

## when/otherwise: per-row branching, first match wins

In [ ]:
sized = orders.withColumn(
    "order_size",
    when(col("quantity") >= 3, "bulk")
    .when(col("quantity") == 2, "pair")
    .otherwise("single"),
)
sized.groupBy("order_size").count().show()

# Where did the null-quantity rows land? (They matched NO when-condition.)
sized.filter(col("quantity").isNull()).select("order_id", "quantity", "order_size").show()

In [ ]:
# Missing otherwise = silent nulls:
no_else = orders.withColumn("size", when(col("quantity") >= 3, "bulk"))
no_else.filter(col("size").isNull()).count()   # everything that wasn't 'bulk'

## expr(): same tree from a SQL string

In [ ]:
a = orders.withColumn("line_total", col("quantity") * col("unit_price"))
b = orders.withColumn("line_total", expr("quantity * unit_price"))

# identical results — and (Module 11 preview) identical plans:
print(a.schema == b.schema)
orders.filter(expr("country IN ('IN','UK') AND quantity >= 2")).count()

## The F namespace: built-ins before custom Python, always

In [ ]:
orders.select(
    "order_id",
    F.upper(col("product")).alias("product_uc"),
    F.round(col("unit_price") * 1.18, 2).alias("price_with_tax"),
    F.year(col("order_date")).alias("yr"),
    F.month(col("order_date")).alias("mo"),
).show(5)

## Exercises

1. Price band column: `budget` < 15 ≤ `mid` < 60 ≤ `premium` — once with `when`, once with `expr("CASE WHEN ...")`. Same `groupBy` counts?
2. Reusable expression: define `is_domestic = col("country") == lit("IN")` and use it in a filter AND (with `.alias`) as a boolean column in a select.
3. Using only `F` functions: extract the weight suffix from products like `Espresso Beans 1kg` / `Green Tea 200g` (`F.regexp_extract(col, r'(\\d+k?g)', 1)`). How many products have one?
4. What does `when(col("quantity") > 1, "multi")` alone (no otherwise) produce for order 1012? Verify.

In [ ]:
# your exercise code here
